<small><font color=gray>Notebook author: Shul Ilya Andreevich ©2026</font></small><hr style="margin:0;background-color:silver">

**Sneaker Resale Hypothesis Testing Notebook**

### Introduction
Secondary markets for clothing apparel specifically streetwear and limited edition sneakers has always been an under the radar topic.
Being a representative of a younger generation I have vast interest in such items thus I wanted to shed light on this topic as this financial market has been bustling throughout the last 8 years.
Indeed this is a market.
Transactions occur continuously prices fluctuate and participants react to scarcity and information with limited drops only increasing demand.
StockX pioneered the scene of reselling and turned demand into a truly one of a kind market where everything from number of pairs sold to demand for specific sizes affects pricing of a seemingly mundane shoe.
In this economy the key financial indicator is the premium of resale price which is essentially a surplus over retail price.

Main question :
Which factors explain cross-sectional differences and time dynamics of resale price premiums and under what conditions does the premium increase with time since release.

### Target variables
Premium<sub>i</sub> = (P<sup>sale</sup><sub>i</sub> - P<sup>retail</sup><sub>i</sub>) / P<sup>retail</sup><sub>i</sub>  
DaysSinceRelease<sub>i</sub> = OrderDate<sub>i</sub> - ReleaseDate<sub>i</sub>

### Hypotheses tested in this notebook
1) Brand premium story : Off-White should carry a higher premium rate than Yeezy.
2) Time decay story : premium should move down as days since release move up.
3) Regional demand story : premium distribution should be different across major buyer regions.

<hr color=green size=40>

<strong><font color=green size=5>Main</font></strong>

<font color=green><h3><b>$\alpha$. Imports and setup</b></h3></font>

In [ ]:
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

RANDOM_STATE = 42
ALPHA = 0.05

<font color=green><h3><b>$\beta$. Timer block</b></h3></font>

In [63]:
class Timer:
    def __init__(self):
        self.t0 = time.time()

    def show_time(self):
        dt = time.time() - self.t0
        print(f"Elapsed time: {dt:.2f} seconds")

tmr = Timer()

<font color=green><h3><b>$\gamma$. Data loading and preprocessing</b></h3></font>

In [65]:
DATA_PATH = 'StockX-Data-Contest-2019-3 2.csv'

df = pd.read_csv(DATA_PATH)

for col in ['Sale Price', 'Retail Price']:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(r'[$,]', '', regex=True),
        errors='coerce'
    )

df['Brand'] = df['Brand'].astype(str).str.strip()
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%y', errors='coerce')
df['Release Date'] = pd.to_datetime(df['Release Date'], format='%m/%d/%y', errors='coerce')

df['Days Since Release'] = (df['Order Date'] - df['Release Date']).dt.days
df['Premium $'] = df['Sale Price'] - df['Retail Price']
df['Premium %'] = 100 * df['Premium $'] / df['Retail Price']

analysis_df = df.dropna(subset=['Brand', 'Buyer Region', 'Days Since Release', 'Premium %']).copy()

print('Rows in raw data:', len(df))
print('Rows used for hypothesis tests:', len(analysis_df))
print('Brand counts:')
print(analysis_df['Brand'].value_counts())
print('Preview:')
display(analysis_df[['Brand', 'Sneaker Name', 'Sale Price', 'Retail Price', 'Premium %', 'Days Since Release', 'Buyer Region']].head())

Rows in raw data: 99956
Rows used for hypothesis tests: 99956
Brand counts:
Brand
Yeezy        72162
Off-White    27794
Name: count, dtype: int64
Preview:


,Brand,Sneaker Name,Sale Price,Retail Price,Premium %,Days Since Release,Buyer Region
0,Yeezy,Adidas-Yeezy-Boost-350-Low-V2-Beluga,1097,220,398.636364,342,California
1,Yeezy,Adidas-Yeezy-Boost-350-V2-Core-Black-Copper,685,220,211.363636,282,California
2,Yeezy,Adidas-Yeezy-Boost-350-V2-Core-Black-Green,690,220,213.636364,282,California
3,Yeezy,Adidas-Yeezy-Boost-350-V2-Core-Black-Red,1075,220,388.636364,282,Kentucky
4,Yeezy,Adidas-Yeezy-Boost-350-V2-Core-Black-Red-2017,828,220,276.363636,202,Rhode Island


<font color=green><h3><b>$\delta$. Plotly visual diagnostics</b></h3></font>

In [66]:
vis_df = analysis_df.copy()

q_low, q_high = vis_df['Premium %'].quantile([0.01, 0.99])
vis_df['Premium % (winsorized)'] = vis_df['Premium %'].clip(q_low, q_high)

COLOR_A = '#00d4ff'
COLOR_B = '#ff7a18'
DARK_BG = '#0b0f14'

pio.templates.default = 'plotly_dark'


def show_graph(fig, name):
    try:
        from IPython.display import HTML, display
        html = fig.to_html(full_html=False, include_plotlyjs='cdn')
        display(HTML(html))
    except Exception as e:
        out_dir = Path('plotly_exports')
        out_dir.mkdir(exist_ok=True)
        out_path = out_dir / f'{name}.html'
        fig.write_html(str(out_path), include_plotlyjs='cdn', full_html=True, auto_open=False)
        print(f'Inline display unavailable. Saved chart file: {out_path.resolve()}')
        print(f'Render error: {e}')

<font color=green><h4><b>Graph 1 (Brand Premium Distribution)</b></h4></font>

In [75]:
brand_sample = vis_df.groupby('Brand', group_keys=False).apply(
    lambda g: g.sample(min(6000, len(g)), random_state=RANDOM_STATE)
).reset_index(drop=True)

fig_brand = px.violin(
    brand_sample,
    x='Brand',
    y='Premium % (winsorized)',
    color='Brand',
    box=True,
    points='all',
    hover_data=['Shoe Size', 'Buyer Region'],
    color_discrete_map={'Yeezy': COLOR_A, 'Off-White': COLOR_B},
    title='Brand premium distribution '
)

fig_brand.update_traces(marker=dict(size=3, opacity=0.24), jitter=0.22)
fig_brand.update_layout(
    template='plotly_dark',
    legend_title_text='Brand',
    height=560,
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    font=dict(color='#e5e7eb')
)
show_graph(fig_brand, 'graph_1_brand_distribution')

<font color=green><h4><b>Graph 2 (Time Dynamics of Premium)</b></h4></font>

In [68]:
time_sample = vis_df.groupby('Brand', group_keys=False).apply(
    lambda g: g.sample(min(5000, len(g)), random_state=RANDOM_STATE)
).reset_index(drop=True)

fig_time = px.scatter(
    time_sample,
    x='Days Since Release',
    y='Premium % (winsorized)',
    color='Brand',
    opacity=0.18,
    hover_data=['Sneaker Name', 'Shoe Size', 'Buyer Region'],
    color_discrete_map={'Yeezy': COLOR_A, 'Off-White': COLOR_B},
    title='Premium vs Days Since Release '
)

for brand, color in [('Yeezy', COLOR_A), ('Off-White', COLOR_B)]:
    sub = vis_df.loc[vis_df['Brand'] == brand, ['Days Since Release', 'Premium % (winsorized)']].copy()
    daily = (
        sub.groupby('Days Since Release', as_index=False)['Premium % (winsorized)']
        .median()
        .sort_values('Days Since Release')
    )
    daily['smoothed'] = daily['Premium % (winsorized)'].rolling(window=21, center=True, min_periods=6).mean()

    fig_time.add_trace(
        go.Scatter(
            x=daily['Days Since Release'],
            y=daily['smoothed'],
            mode='lines',
            name=f'{brand} rolling median',
            line=dict(color=color, width=3)
        )
    )

fig_time.update_layout(
    template='plotly_dark',
    height=600,
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    font=dict(color='#e5e7eb')
)
show_graph(fig_time, 'graph_2_time_dynamics')

<font color=green><h4><b>Graph 3 (Region Premium Heatmap)</b></h4></font>

In [69]:
top10_regions = vis_df['Buyer Region'].value_counts().head(10).index
heat = (
    vis_df[vis_df['Buyer Region'].isin(top10_regions)]
    .pivot_table(index='Buyer Region', columns='Brand', values='Premium % (winsorized)', aggfunc='median')
    .reindex(top10_regions)
)

fig_heat = go.Figure(
    data=go.Heatmap(
        z=heat.values,
        x=heat.columns,
        y=heat.index,
        colorscale=[[0.0, COLOR_A], [1.0, COLOR_B]],
        colorbar=dict(title='Median Premium %'),
        hovertemplate='Region: %{y}<br>Brand: %{x}<br>Median Premium: %{z:.2f}%<extra></extra>'
    )
)

fig_heat.update_layout(
    title='Median premium by region and brand heatmap',
    template='plotly_dark',
    height=560,
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    font=dict(color='#e5e7eb')
)
show_graph(fig_heat, 'graph_3_region_heatmap')

<font color=green><h4><b>Graph 4 (3D Market Map)</b></h4></font>

In [70]:
three_d = vis_df[['Days Since Release', 'Shoe Size', 'Premium % (winsorized)', 'Brand', 'Sneaker Name', 'Buyer Region']].dropna()
three_d = three_d.sample(min(9000, len(three_d)), random_state=RANDOM_STATE)

fig_3d = px.scatter_3d(
    three_d,
    x='Days Since Release',
    y='Shoe Size',
    z='Premium % (winsorized)',
    color='Brand',
    hover_data=['Sneaker Name', 'Buyer Region'],
    color_discrete_map={'Yeezy': COLOR_A, 'Off-White': COLOR_B},
    title='3D market map : time, size, premium'
)

fig_3d.update_traces(marker=dict(size=3, opacity=0.36))
fig_3d.update_layout(
    template='plotly_dark',
    height=650,
    paper_bgcolor=DARK_BG,
    font=dict(color='#e5e7eb'),
    scene=dict(
        xaxis=dict(backgroundcolor=DARK_BG, gridcolor='rgba(255,255,255,0.13)', zerolinecolor='rgba(255,255,255,0.16)'),
        yaxis=dict(backgroundcolor=DARK_BG, gridcolor='rgba(255,255,255,0.13)', zerolinecolor='rgba(255,255,255,0.16)'),
        zaxis=dict(backgroundcolor=DARK_BG, gridcolor='rgba(255,255,255,0.13)', zerolinecolor='rgba(255,255,255,0.16)')
    )
)
show_graph(fig_3d, 'graph_4_market_map_3d')

<font color=green><h3><b>$\epsilon$. Hypothesis 1: Brand effect</b></h3></font>

I am testing whether brand identity alone can create a large premium gap.
H0 : mean premium rate is the same for Off-White and Yeezy.
H1 : mean premium rate is different and expected to be higher for Off-White.

Method :
Welch two sample t-test is used because group variances are different.

In [71]:
off_white = analysis_df.loc[analysis_df['Brand'] == 'Off-White', 'Premium %']
yeezy = analysis_df.loc[analysis_df['Brand'] == 'Yeezy', 'Premium %']

h1_test = stats.ttest_ind(off_white, yeezy, equal_var=False)

def cohen_d(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    n1, n2 = len(a), len(b)
    s1, s2 = np.var(a, ddof=1), np.var(b, ddof=1)
    pooled = ((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)
    return (np.mean(a) - np.mean(b)) / np.sqrt(pooled)

h1_effect = cohen_d(off_white, yeezy)

print('Off-White mean premium %:', round(off_white.mean(), 2))
print('Yeezy mean premium %    :', round(yeezy.mean(), 2))
print('Welch t-stat:', round(h1_test.statistic, 4))
print('p-value     :', h1_test.pvalue)
print("Cohen's d   :", round(h1_effect, 4))

if h1_test.pvalue < ALPHA:
    print('Decision: Reject H0. Brand effect is statistically significant.')
else:
    print('Decision: Fail to reject H0 at alpha = 0.05.')

Off-White mean premium %: 282.87
Yeezy mean premium %    : 63.95
Welch t-stat: 188.9376
p-value     : 0.0
Cohen's d   : 1.9099
Decision: Reject H0. Brand effect is statistically significant.


<font color=green><h3><b>$\zeta$. Hypothesis 2: Time effect</b></h3></font>

I am testing whether time since release carries a monotonic signal for premium.
H0 : there is no monotonic relation between Days Since Release and Premium %.
H1 : there is a monotonic relation and expected direction is negative.

Method :
Spearman rank correlation.

In [72]:
h2_test = stats.spearmanr(analysis_df['Days Since Release'], analysis_df['Premium %'])

print('Spearman rho:', round(h2_test.statistic, 4))
print('p-value     :', h2_test.pvalue)

if h2_test.pvalue < ALPHA:
    direction = 'negative' if h2_test.statistic < 0 else 'positive'
    print(f'Decision: Reject H0. Relationship is statistically significant and {direction}.')
else:
    print('Decision: Fail to reject H0 at alpha = 0.05.')

Spearman rho: -0.1936
p-value     : 0.0
Decision: Reject H0. Relationship is statistically significant and negative.


<font color=green><h3><b>$\eta$. Hypothesis 3: Region effect</b></h3></font>

I am testing whether location level demand differences are strong enough to shift premium distributions.
H0 : premium distributions are equal across major buyer regions.
H1 : at least one major region has a different premium distribution.

Method :
Kruskal-Wallis on top 6 regions by transaction count because the premium distribution is not normal.

In [73]:
top_regions = analysis_df['Buyer Region'].value_counts().head(6).index.tolist()
region_samples = [analysis_df.loc[analysis_df['Buyer Region'] == r, 'Premium %'] for r in top_regions]

h3_test = stats.kruskal(*region_samples)

region_summary = (
    analysis_df[analysis_df['Buyer Region'].isin(top_regions)]
    .groupby('Buyer Region')['Premium %']
    .agg(['count', 'mean', 'median'])
    .sort_values('mean', ascending=False)
)

print('Top regions used in test:', top_regions)
print('Kruskal-Wallis H:', round(h3_test.statistic, 4))
print('p-value         :', h3_test.pvalue)

if h3_test.pvalue < ALPHA:
    print('Decision: Reject H0. Region-level distributions differ significantly.')
else:
    print('Decision: Fail to reject H0 at alpha = 0.05.')

display(region_summary)

Top regions used in test: ['California', 'New York', 'Oregon', 'Florida', 'Texas', 'New Jersey']
Kruskal-Wallis H: 530.4315
p-value         : 2.1502288012312697e-112
Decision: Reject H0. Region-level distributions differ significantly.


,count,mean,median
Buyer Region,,,
California,19349,143.817587,78.636364
Oregon,7681,139.961596,85.000000
New Jersey,4720,125.945409,70.454545
Florida,6376,125.570776,70.454545
New York,16525,122.140293,69.545455
Texas,5876,107.399904,63.636364


<font color=green><h3><b>$\theta$. Final hypothesis summary</b></h3></font>

In [74]:
summary = pd.DataFrame([
    {
        'Hypothesis': 'H1: Brand effect (Off-White vs Yeezy)',
        'Test': 'Welch t-test',
        'Statistic': float(h1_test.statistic),
        'P-value': float(h1_test.pvalue),
        'Reject H0 (alpha=0.05)': bool(h1_test.pvalue < ALPHA),
    },
    {
        'Hypothesis': 'H2: Days Since Release vs Premium %',
        'Test': 'Spearman correlation',
        'Statistic': float(h2_test.statistic),
        'P-value': float(h2_test.pvalue),
        'Reject H0 (alpha=0.05)': bool(h2_test.pvalue < ALPHA),
    },
    {
        'Hypothesis': 'H3: Regional differences (top-6)',
        'Test': 'Kruskal-Wallis',
        'Statistic': float(h3_test.statistic),
        'P-value': float(h3_test.pvalue),
        'Reject H0 (alpha=0.05)': bool(h3_test.pvalue < ALPHA),
    },
])

summary

,Hypothesis,Test,Statistic,P-value,Reject H0 (alpha=0.05)
0,H1: Brand effect (Off-White vs Yeezy),Welch t-test,188.937608,0.000000e+00,True
1,H2: Days Since Release vs Premium %,Spearman correlation,-0.193604,0.000000e+00,True
2,H3: Regional differences (top-6),Kruskal-Wallis,530.431478,2.150229e-112,True


<font color=green><h3><b>$\iota$. ML setup and model selection</b></h3></font>

I am now moving from hypothesis testing to prediction. At this point the question is not only whether these factors matter statistically but whether they are strong enough to forecast premium on future transactions. I keep the split chronological so the last block stays untouched until the final step and I compare simple baselines against stronger nonlinear models before choosing the final specification.

In [ ]:
import json
import subprocess
from IPython.display import Image, display

ML_PYTHON = '/Library/Frameworks/Python.framework/Versions/3.12/bin/python3'
RUN_ML_REBUILD = False

if RUN_ML_REBUILD:
    subprocess.run([ML_PYTHON, 'build_sneaker_research_assets.py'], check=True)

with open('tables/research_summary.json') as f:
    ml_summary = json.load(f)

split_table = pd.read_csv('tables/data_split_summary.csv')

ml_overview = pd.DataFrame([
    {'Item': 'Selected regression model', 'Value': ml_summary['best_regression_model']},
    {'Item': 'Ridge alpha', 'Value': ml_summary['ridge_alpha']},
    {'Item': 'Selected classification model', 'Value': ml_summary['best_classification_model']},
    {'Item': 'Logistic C', 'Value': ml_summary['logistic_c']},
    {'Item': 'Regression RMSE 95% CI', 'Value': f"[{ml_summary['regression_test_metrics_ci']['RMSE'][0]:.2f}, {ml_summary['regression_test_metrics_ci']['RMSE'][1]:.2f}]"},
    {'Item': 'Classification ROC-AUC 95% CI', 'Value': f"[{ml_summary['classification_test_metrics_ci']['ROC_AUC'][0]:.3f}, {ml_summary['classification_test_metrics_ci']['ROC_AUC'][1]:.3f}]"},
])

display(split_table)
display(ml_overview)

<font color=green><h3><b>$\kappa$. Regression models</b></h3></font>

For the regression task I predict `Premium %` directly. I compare a plain linear baseline, Ridge regression and tree based models. Ridge is selected because it is the cleanest and most stable model on the validation window and it stays interpretable which matters for this project.

In [ ]:
reg_validation = pd.read_csv('tables/regression_validation_results.csv').round(3)
reg_test = pd.read_csv('tables/regression_test_results.csv').round(3)
reg_selected = reg_test.loc[reg_test['Model'] == 'Ridge'].reset_index(drop=True)
reg_diagnostics = pd.read_csv('tables/regression_diagnostics.csv').round(3)

display(reg_validation)
display(reg_selected)
display(reg_diagnostics)

In [ ]:
reg_points = pd.read_csv('tables/regression_holdout_predictions.csv')
reg_points = reg_points.sample(min(3000, len(reg_points)), random_state=RANDOM_STATE)

lo = min(reg_points['Actual Premium %'].min(), reg_points['Predicted Premium %'].min())
hi = max(reg_points['Actual Premium %'].max(), reg_points['Predicted Premium %'].max())

fig_reg = px.scatter(
    reg_points,
    x='Actual Premium %',
    y='Predicted Premium %',
    opacity=0.28,
    color_discrete_sequence=[COLOR_A],
    title='Graph 5 (Regression Predicted vs Actual Premium)'
)

fig_reg.update_traces(marker=dict(size=4))
fig_reg.add_trace(
    go.Scatter(
        x=[lo, hi],
        y=[lo, hi],
        mode='lines',
        name='Perfect fit',
        line=dict(color=COLOR_B, width=3)
    )
)

fig_reg.update_layout(
    template='plotly_dark',
    height=560,
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    font=dict(color='#e5e7eb'),
    xaxis_title='Actual Premium %',
    yaxis_title='Predicted Premium %'
)
show_graph(fig_reg, 'graph_5_regression_actual_vs_predicted')

reg_imp = pd.read_csv('tables/regression_feature_importance.csv')
reg_imp = reg_imp.iloc[::-1].reset_index(drop=True)
reg_colors = [COLOR_A if i % 2 == 0 else COLOR_B for i in range(len(reg_imp))]

fig_reg_imp = go.Figure(
    go.Bar(
        x=reg_imp['Importance'],
        y=reg_imp['Feature'],
        orientation='h',
        marker=dict(color=reg_colors),
        hovertemplate='%{y}<br>Importance: %{x:.3f}<extra></extra>'
    )
)

fig_reg_imp.update_layout(
    template='plotly_dark',
    height=540,
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    font=dict(color='#e5e7eb'),
    title='Graph 6 (Regression Feature Importance)',
    xaxis_title='Mean decrease in R² when permuted',
    yaxis_title=''
)
show_graph(fig_reg_imp, 'graph_6_regression_feature_importance')

<font color=green><h3><b>$\lambda$. Classification models</b></h3></font>

For the classification task I define a high premium event as `Premium % >= 100`. In simple terms this is a shoe that resells for at least double retail. This is useful because it turns the market question into a very practical yes or no decision and lets me evaluate whether scarcity signals are strong enough to identify exceptional resale outcomes.

In [ ]:
class_validation = pd.read_csv('tables/classification_validation_results.csv').round(3)
class_test = pd.read_csv('tables/classification_test_results.csv').round(3)
class_selected = class_test.loc[class_test['Model'] == 'HistGradientBoostingClassifier'].reset_index(drop=True)
class_importance = pd.read_csv('tables/classification_feature_importance.csv').round(3)

display(class_validation)
display(class_selected)
display(class_importance)

In [ ]:
class_auc = class_test[['Model', 'ROC_AUC']].copy()
class_auc = class_auc.sort_values('ROC_AUC', ascending=True).reset_index(drop=True)
class_colors = [COLOR_A if i % 2 == 0 else COLOR_B for i in range(len(class_auc))]

fig_class_auc = go.Figure(
    go.Bar(
        x=class_auc['ROC_AUC'],
        y=class_auc['Model'],
        orientation='h',
        marker=dict(color=class_colors),
        hovertemplate='%{y}<br>ROC-AUC: %{x:.3f}<extra></extra>'
    )
)

fig_class_auc.update_layout(
    template='plotly_dark',
    height=500,
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    font=dict(color='#e5e7eb'),
    title='Graph 7 (Classification ROC-AUC by Model)',
    xaxis_title='ROC-AUC',
    yaxis_title=''
)
show_graph(fig_class_auc, 'graph_7_classification_auc')

class_imp = pd.read_csv('tables/classification_feature_importance.csv')
class_imp = class_imp.iloc[::-1].reset_index(drop=True)
class_colors = [COLOR_A if i % 2 == 0 else COLOR_B for i in range(len(class_imp))]

fig_class_imp = go.Figure(
    go.Bar(
        x=class_imp['Importance'],
        y=class_imp['Feature'],
        orientation='h',
        marker=dict(color=class_colors),
        hovertemplate='%{y}<br>Importance: %{x:.3f}<extra></extra>'
    )
)

fig_class_imp.update_layout(
    template='plotly_dark',
    height=540,
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
    font=dict(color='#e5e7eb'),
    title='Graph 8 (Classification Feature Importance)',
    xaxis_title='Mean decrease in ROC-AUC when permuted',
    yaxis_title=''
)
show_graph(fig_class_imp, 'graph_8_classification_feature_importance')

<font color=green><h3><b>$\mu$. Conclusion</b></h3></font>

Final conclusion : All three hypotheses were supported and the ML results reinforce the same story. H1 means brand power is a key pricing force because Off-White keeps a much larger premium than Yeezy. H2 means hype decays with time and that the market pays the highest premium closer to release. H3 means geography matters and demand is not identical across regions.

The predictive results matter for the same reason. Ridge regression explains around 67 percent of future premium variation on the holdout sample and the boosted classifier reaches ROC-AUC near 0.973 for transactions where resale price at least doubles retail. This means the premium is not only explainable after the fact but also predictable to a meaningful degree. In short this market looks chaotic from the outside but once brand timing and scarcity are modeled directly the structure becomes much clearer.

In [ ]:
tmr.show_time()